# 08 - Pass 4: training with the generated headline pairs (one commit, GPU)

This notebook trains every Pass 4 run in **one Save & Run All (Commit)**. The code lives in the GitHub repo (`src/gainsplit/`); the notebook fetches it, checks the data on the CPU first, then trains two runs at a time (one per GPU).

**What Pass 4 asks** (vault: Experiment Plan, "Pass 4 plan"): do the LLM-written sarcastic/serious headline pairs help a BERT sarcasm classifier with few labels, does the *pairing* matter, does the judge filter matter, and does the model lean less on outlet words?

| Arm | What it trains on |
|---|---|
| A10 | the labeled headlines (L) + one batch of generated pairs per step |
| A11 | A10 + FGM (the adversarial push from A7) |
| A12 | control: one side of each pair only (no topic-matched partner) |
| A13 | A10 with the judge-filtered pairs |
| A14 | control for A13: a random subset with the same pairs per model |
| A15 | generated pairs only, no human labels (same number of steps as 10%) |

Plus re-runs of A1 (10%, with explanations) and of A1 and A7 at 1% (adding the masked test sets). Re-runs must give exactly the Pass 1 / Pass 2b scores: that checks the new code did not change the old path.

**Before you run, in the right-hand panel:**
- **Input:** the new version of **`gainsplit-processed`** that also holds `pairs_train.csv`, `manifest_pairs.json`, `pairs_train_filtered.csv` and `manifest_judge_v2.json`.
- **Internet: on** (code from GitHub, BERT from Hugging Face; no token needed).
- **Accelerator: GPU T4 x2**, then **Save Version -> Save & Run All (Commit)**.

**Saving GPU quota.** Edit with the accelerator set to **None**. Switch to GPU T4 x2 only to start the commit, then **stop the interactive session** (the power button at the top right): an open interactive session with a GPU uses quota too, in parallel with the commit. Step 3 checks every file on the CPU before any training, so a missing or wrong file stops the run within minutes.

**Time:** about 1.3-1.5 h for `pass4` (75 runs). Details in step 4.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about a minute)
The repo's tests train a tiny random BERT for a few steps on made-up data, including the pair arms, the pair batches and the explanation step. They prove the pieces fit together before real GPU time is spent. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the data on the CPU (pre-flight)
1. Finds `gainsplit-processed` and checks every file against `manifest.json` and `manifest_pass3.json` (masked test sets, cue words).
2. Loads each pair set the runs will use and checks it: SHA-256 against its manifest, exactly one sarcastic and one serious row per seed, seeds from the training split only, no generated headline equal to a dev or test headline, and the filtered file made from this exact `pairs_train.csv`.
3. Prints the size of each set. Expected: `unfiltered` 44,834 rows (22,417 pairs); `filtered` and `matched` 34,630 rows each with the same pairs per model (Gemma 7,533, Mistral 4,986, Qwen 4,796); `unpaired` 22,417 rows (one side per seed).

Any problem raises an error here, before training starts.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS

PLAN = "pass4"          # or "pass4_full100" (optional extra, see step 4)
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = PLANS[PLAN]()

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
display(preflight(configs, DATA_DIR))

## 4. The list of runs and the time estimate
`pass4` = 75 runs, seeds 0-4 each, test scored, masked test sets scored:
- **10%**: A11, A10 (+ explanations), A12, A13, A14, A1 re-run (+ explanations), A15.
- **1%**: A10 with 3x the steps (does 1% need longer training?), A11, A10, A12, A13, A14, A1 and A7 re-runs (tag `mask`).

Estimated run time (from measured Pass 1-3 times; a pair arm costs about as much as a helper arm, which also adds a second batch): 10% about 2.1 min per run (A11 about 3.3, A1 1.4, explanations add about 1), 1% about 0.7 min. Total about 120 run-minutes plus start-up, so **about 70-80 minutes on two GPUs**, plus about 5 minutes for steps 1-3.

`pass4_full100` (optional, not in `pass4`): A10 at 100% labels, 5 runs of about 20 min each, so about 50 more minutes on two GPUs. Run it later as a second commit only if quota is left; finished runs are never repeated.

In [ ]:
for c in configs:
    print(f"{c.run_id:24s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} eps={c.fgm_epsilon:g} "
          f"step_mult={c.step_mult} pair_batch={c.pair_batch} explain={c.explain}")
print(len(configs), "runs")

## 5. Run
One run per GPU; when a GPU frees up, the next run starts. Each run's log is in `results/<plan>/logs/<run_id>.log`. If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## 6. First look (the full analysis runs on the CPU, notebook 09)
- Mean test macro-F1 (points) per arm and budget, next to the baselines from Passes 1-2 (read from the repo clone).
- **Re-run check:** the A1 and A7 re-runs must equal the earlier scores (difference 0). A non-zero difference means the new code changed the old path: report it before reading anything else.
- Explanations: mean share of attribution on outlet cue words (A1 vs A10 at 10%).

In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for f in sorted(Path(OUT_DIR).glob("runs/*/result.json")):
    r = json.loads(f.read_text()); c = r["config"]
    rows.append({"arm": c["arm"], "budget": c["budget"], "seed": c["seed"], "tag": c.get("tag", ""),
                 "test_f1": r["scores"]["test"]["macro_f1"] * 100,
                 "cuemask_f1": r["scores"].get("test_cuemask", {}).get("macro_f1", float("nan")) * 100,
                 "isarc_auc": r["scores"]["isarc_test"]["roc_auc"] * 100,
                 "cue_share": (r.get("explain") or {}).get("mean_cue_share"), "minutes": r["minutes"]})
res = pd.DataFrame(rows)
display(res.groupby(["budget", "arm", "tag"])[["test_f1", "cuemask_f1", "isarc_auc", "cue_share", "minutes"]]
           .mean().round(3))

base = []
for p in ("pass1", "pass2b"):
    s = Path(REPO_DIR) / "results" / p / "summary.csv"
    if s.exists():
        base.append(pd.read_csv(s))
if base:
    base = pd.concat(base)
    display(base[base["arm"].isin(["A1", "A7"]) & base["budget"].isin([1, 10])]
            .assign(test_f1=lambda d: d["test_macro_f1"] * 100)
            .groupby(["budget", "arm"])["test_f1"].mean().round(3).to_frame("baseline test_f1 (Pass 1/2b)"))
    chk = res[res["tag"].isin(["explain", "mask"])].merge(
        base[["arm", "budget", "seed", "test_macro_f1"]], on=["arm", "budget", "seed"], how="left")
    chk["diff"] = (chk["test_f1"] - chk["test_macro_f1"] * 100).abs()
    display(chk[["arm", "budget", "seed", "tag", "test_f1", "diff"]].round(6))
    print("re-run check:", "PASS" if (chk["diff"] < 1e-6).all() else "DIFFERENCE FOUND - report it")

## 7. Download
Zips the results folder. A commit keeps it in the Output tab: download `results_<plan>.zip` and send it with the log.

In [ ]:
import shutil
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")